# Exploratory Data Analysis — Walmart Mexico Cash Demand

This notebook explores the raw transaction data and validates the statistical
assumptions that motivated the modeling choices:

1. Distribution of cash demand (justifies lognormal model)
2. Weekly seasonality (justifies day-of-week features)
3. Payday (quincena) effect (justifies `is_payday` feature)
4. Stationarity across stores
5. STL decomposition of a representative store


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from walmart_cash_forecast.data.loader import DataLoader
from walmart_cash_forecast.features.aggregator import StoreAggregator
from walmart_cash_forecast.features.imputer import CashImputer
from walmart_cash_forecast.stats.reporter import StatAnalyzer

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../Prueba_Tecnica_DS')

In [ ]:
transactions, stores, calendar = DataLoader(DATA_DIR).load()
print(f'Transactions: {transactions.shape}')
print(f'Stores: {stores.shape}')
transactions.head()

In [ ]:
# Aggregate to store-day level (cashiers share one float per store)
panel = StoreAggregator().aggregate(transactions)
panel['day_of_week'] = panel['date'].dt.dayofweek
panel['is_payday'] = panel['date'].dt.day.isin([15]) | (
    panel['date'] == panel['date'].dt.to_period('M').dt.to_timestamp('M')
)
panel = CashImputer().fit(panel).transform(panel)
print(f'Panel shape: {panel.shape}, nulls: {panel["amount_cash"].isna().sum()}')
panel.head()

## 1. Distribution of daily cash demand

We expect a right-skewed distribution — a log-normal is a natural choice for
positive-valued, multiplicatively driven quantities like retail cash demand.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(panel['amount_cash'] / 1_000, bins=60, edgecolor='none')
axes[0].set(title='Cash demand distribution', xlabel='MXN (thousands)', ylabel='Count')
axes[1].hist(np.log1p(panel['amount_cash']), bins=60, edgecolor='none', color='steelblue')
axes[1].set(title='log(1 + cash demand)', xlabel='log1p(MXN)', ylabel='Count')
plt.tight_layout()
plt.savefig('../stats_report/distribution.png', dpi=150)
plt.show()

## 2. Weekly seasonality

Retailers exhibit strong day-of-week effects. We expect weekends and paydays
to have higher demand.

In [ ]:
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_agg = panel.groupby('day_of_week')['amount_cash'].median() / 1_000
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dow_labels, dow_agg.values, color='steelblue', edgecolor='none')
ax.set(title='Median cash demand by day of week', ylabel='MXN (thousands)')
plt.tight_layout()
plt.savefig('../stats_report/day_of_week.png', dpi=150)
plt.show()

## 3. Payday (quincena) effect

Mexico's quincena paydays (day 15 and last day of month) should produce a
significant increase in cash demand.

In [ ]:
from scipy import stats

payday_cash = panel.loc[panel['is_payday'], 'amount_cash'].values
nonpayday_cash = panel.loc[~panel['is_payday'], 'amount_cash'].values

stat, pval = stats.mannwhitneyu(payday_cash, nonpayday_cash, alternative='greater')
effect_size = (np.median(payday_cash) - np.median(nonpayday_cash)) / np.median(nonpayday_cash)

print(f'Mann-Whitney U p-value: {pval:.4f}')
print(f'Effect size (relative median lift): {effect_size:.2%}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.violinplot([nonpayday_cash / 1_000, payday_cash / 1_000], positions=[0, 1], showmedians=True)
ax.set(xticks=[0, 1], xticklabels=['Non-payday', 'Payday'],
       ylabel='MXN (thousands)', title=f'Payday effect (p={pval:.4f}, lift={effect_size:.1%})')
plt.tight_layout()
plt.savefig('../stats_report/payday_effect.png', dpi=150)
plt.show()

## 4. Full statistical report

Run all four analyses (distribution, stationarity, STL, payday) and save JSON.

In [ ]:
import json
report_dir = Path('../stats_report')
report_dir.mkdir(exist_ok=True)

# StatAnalyzer requires cash_transactions column; add it from transactions if available
if 'cash_transactions' not in panel.columns:
    tx_agg = transactions.groupby(['store_id', 'date'], as_index=False)['cash_transactions'].sum()
    panel = panel.merge(tx_agg, on=['store_id', 'date'], how='left')

report = StatAnalyzer().run(panel, report_dir)
print(json.dumps(report, indent=2, default=str))

## 5. STL decomposition

Seasonal-Trend via LOESS (period=7) on a representative store.

In [ ]:
from statsmodels.tsa.seasonal import STL

store_id = panel['store_id'].iloc[0]
store_series = (
    panel[panel['store_id'] == store_id]
    .sort_values('date')['amount_cash']
    .reset_index(drop=True)
)

stl = STL(store_series, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 8)
fig.suptitle(f'STL Decomposition — Store {store_id}', y=1.01)
plt.tight_layout()
plt.savefig('../stats_report/stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()